# Simple RAG Training Pipeline

This notebook provides a streamlined 4-step process to generate synthetic QA data and launch a training job:

1. **Point to dataset** - Chunk and upload documents
2. **Create synthetic QA** - Generate question-answer pairs
3. **Create env** - Bundle and upload search environment
4. **Run training job** - Launch the training

## Setup

Install dependencies and configure API credentials.

In [ ]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgft-io/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev]

In [1]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
# Configuration

# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = "sk_IYfzBWETfgLrxzXLcXfDNvruSnlNLOGHxnMrlhDrOtczzpBdyTMjYeKPcZEMFkNS"
BASE_URL = "https://app.cgft.io"

# Dataset configuration

# this should point to a local directory containing the documents you want to use for dataset generation.
# go to docs.cgft.io/rag/synthetic_datagen for sample documents you can use.
DOCS_PATH = "./samples/posthog/docs/"
# this is the name of the corpus that will be created on the cgft platform to store your documents and generated dataset.
CORPUS_NAME = "my-docs"

# QA generation configuration
# the following parameters are used to configure the synthetic question answer generation process. you can experiment with different values to see how they affect the generated dataset. the numbers below are just suggestions to get you started.
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]
# Number of single-hop and multi-hop questions to generate. Single-hop questions can be answered with a single chunk, while multi-hop questions require reasoning over multiple chunks.
NUM_SINGLE_HOP = 15
NUM_MULTI_HOP = 5

## Step 1: Point to Dataset

Chunk your documents and upload to corpus API in one line.

In [3]:
from synthetic_data_prep.corpus import prepare_corpus

corpus, collection, corpus_client = prepare_corpus(
    docs_path=DOCS_PATH,
    corpus_name=CORPUS_NAME,
    api_key=API_KEY,
    base_url=BASE_URL,
)

/Users/girish/Documents/synthetic-data-prep-lib/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunking documents from ./samples/posthog/docs/...
ChunkCollection Summary
Total chunks: 3165
Total files: 1176

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── reddit-ads.mdx (1 chunks)
│   │   ├── s3.mdx (1 chunks)
│   │   ├── temporal.mdx (1 chunks)
│   │   ├── klaviyo.mdx (1 chunks)
│   │       ... 31 more files
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── sql/
│   │   ├── variables.mdx (2 chunks)
│   │   ├── index.mdx (8 chunks)
│   │   ├── data-access.mdx (3 chunks)
│   │   └── useful-functions.mdx (7 chunks)
│   ├── under-the-hood.md (1 chunks)
│   ├── cutting-costs.mdx (3 chunks)
│   ├── join.mdx (2 chunks)
│   ├── tutorials.mdx (1 chunks)
│       ... 5 more files
├── advanced/
│   ├── proxy/
│   │   ├── _s

Uploading chunks: 100%|██████████| 32/32 [00:25<00:00,  1.27batch/s, inserted=3165]


Upload complete! Inserted: 3165


## Step 2: Create Synthetic QA

Generate synthetic question-answer pairs from your corpus. This will take a few minutes, so go get a coffee :)

In [4]:
from synthetic_data_prep.qa_generation import generate_dataset

dataset = generate_dataset(
    corpus=corpus,
    collection=collection,
    corpus_client=corpus_client,
    api_key=API_KEY,
    corpus_description=CORPUS_DESCRIPTION,
    example_queries=EXAMPLE_QUERIES,
    num_single_hop=NUM_SINGLE_HOP,
    num_multi_hop=NUM_MULTI_HOP,
)

Generating corpus summary and example queries...

Generating 15 single-hop QA pairs...


Processing prompts: 100%|██████████| 15/15 [00:33<00:00,  2.26s/it]


QADataset: 26 total data points
  Single-hop: 26
  Multi-hop: 0

Generating 5 multi-hop QA pairs...
Step 1: Generating related queries for 5 chunks...


Processing prompts: 100%|██████████| 5/5 [00:27<00:00,  5.41s/it]



Step 2: Validating 14 chunk pairs for multi-hop QA...


Processing prompts: 100%|██████████| 14/14 [00:37<00:00,  2.71s/it]

QADataset: 22 total data points
  Single-hop: 0
  Multi-hop: 22

Combined dataset: QADataset: 48 total data points
  Single-hop: 26
  Multi-hop: 22

Saved datasets:
  YAML: outputs/qa_dataset.yaml
  JSONL: outputs/qa_dataset.jsonl


## Step 3: Upload Dataset & Create Environment

Upload the dataset and bundle the search environment for training.

In [8]:
from synthetic_data_prep.trainer import upload_dataset, upload_env
from synthetic_data_prep.envs.search_env import SearchEnv
import sys

# Upload dataset
dataset_blob_path = upload_dataset(
    dataset=dataset,
    corpus=corpus,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# Upload environment
env_blob_path, env_args_blob_path = upload_env(
    env_class=SearchEnv,
    constructor_args={
        "api_key": API_KEY,
        "corpus_id": corpus.id,
        "base_url": BASE_URL,
        "dataset_path": f"~/user-data/{dataset_blob_path}",
    },
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[sys.modules[SearchEnv.__module__]],
)

Uploading 48 QA pairs for corpus 09bfcf73-ed4c-4e23-993e-42481bb177e4...
Dataset uploaded to: datasets/search/09bfcf73-ed4c-4e23-993e-42481bb177e4/qa-dataset.jsonl
Bundling environment...
Pickled class size: 15.47 KB
Metadata size: 0.40 KB
Python version: 3.12
Dependencies: ['aiohttp']

Running validation in isolated environment (this may take ~1 min)...
[validator] Creating isolated venv...
[validator] Downloading and installing pip...
[validator] Installing dependencies: ['benchmax', 'aiohttp']
[validator] Dependencies installed successfully.
[validator] Running smoke test...
[validator] OK: SearchEnv instantiated, 1 tools
Isolated validation passed!
Env class uploaded successfully to envs/search/84fd5972/search-env-cls.pkl
Env metadata uploaded successfully to envs/search/84fd5972/search-env-meta.json


In [ ]:
import pathlib
print(sys.modules[SearchEnv.__module__])
print(pathlib)
print(pathlib.__file__)

## Step 4: Run Training Job

Launch the training job with your dataset and environment.

In [6]:
from synthetic_data_prep.trainer import launch_job

experiment_id = launch_job(
    env_cls_blob_path=env_blob_path,
    env_metadata_blob_path=env_args_blob_path,
    api_key=API_KEY,
    base_url=BASE_URL,
)

Launching training experiment...
Training experiment launched! Experiment ID: 136f0649-1fc0-475a-9177-c7d5876a4702


In [7]:
print(env_blob_path)

~/user-data/envs/search/d3f5181b/search-env-cls.pkl


## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:
- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse